# DeepOptima AI: real NAS benchmark on a free GPU

This notebook runs **real** training (not the website simulator) with the DeepOptima search engine from the repo:
evolutionary NSGA-II search vs. a random-search baseline with the **same evaluation budget**, on a CIFAR-10 subset.

1. `Runtime` -> `Change runtime type` -> **T4 GPU**
2. `Runtime` -> `Run all` (about 20-40 minutes with the defaults)
3. Download `deepoptima-results.zip` and open `session.json` on the website: **Dashboard -> Load session**

> Numbers come from short, small-subset training, so treat them as a *relative* comparison between search strategies, not as CIFAR-10 benchmark accuracy.

In [ ]:
!git clone -q https://github.com/Nikhil-creat/deepoptima.git 2>/dev/null || (cd deepoptima && git pull -q)
!pip -q install onnx
import sys, os, time, json, random
sys.path.insert(0, '/content/deepoptima/backend')
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, torchvision
import matplotlib.pyplot as plt
from app.nas.genotype import random_genotype, build_model, count_params, Genotype, DEFAULT_SPACE
from app.nas.evolution import nsga2_select, next_generation
from app.nas.adaptive import adapt_weights

dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU = torch.cuda.get_device_name(0) if dev.type == 'cuda' else 'cpu'
print('device:', dev, GPU)

In [ ]:
# ---- experiment settings (raise POP/GENS/EPOCHS/TRAIN_N for a stronger run) ----
POP, GENS, EPOCHS = 8, 5, 2
TRAIN_N, TEST_N, SEED = 8000, 2000, 42
SPACE = {**DEFAULT_SPACE, 'blocks': ['conv', 'res', 'mbconv'], 'max_blocks': 8}   # add 'vit' for hybrids (slower)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
std = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)

def load(train):
    ds = torchvision.datasets.CIFAR10('/content/data', train=train, download=True)
    n = TRAIN_N if train else TEST_N
    x = torch.tensor(ds.data[:n]).permute(0, 3, 1, 2).float() / 255
    y = torch.tensor(ds.targets[:n])
    return ((x - mean) / std).to(dev), y.to(dev)

Xtr, Ytr = load(True); Xte, Yte = load(False)
print(Xtr.shape, Xte.shape)

In [ ]:
def latency_ms(model, runs=30):
    model.eval(); x = torch.randn(1, 3, 32, 32, device=dev)
    with torch.no_grad():
        for _ in range(10): model(x)
        if dev.type == 'cuda': torch.cuda.synchronize()
        t = time.perf_counter()
        for _ in range(runs): model(x)
        if dev.type == 'cuda': torch.cuda.synchronize()
    return (time.perf_counter() - t) / runs * 1000

def train_eval(g):
    """Train a candidate for EPOCHS and measure accuracy, latency, params. Returns None if it diverges."""
    model = build_model(g).to(dev)
    opt = (torch.optim.AdamW(model.parameters(), g.lr, weight_decay=g.weight_decay) if g.optimizer == 'adamw'
           else torch.optim.SGD(model.parameters(), g.lr, momentum=0.9, weight_decay=g.weight_decay))
    bs, curve = g.batch_size, []
    for ep in range(EPOCHS):
        model.train(); perm = torch.randperm(len(Xtr), device=dev); tot = n = 0
        for i in range(0, len(Xtr), bs):
            idx = perm[i:i + bs]; x, y = Xtr[idx], Ytr[idx]
            if random.random() < 0.5: x = torch.flip(x, dims=[3])        # cheap augmentation
            opt.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(x), y)
            if not torch.isfinite(loss): return None                     # diverged
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
            tot += loss.item() * len(idx); n += len(idx)
        curve.append(tot / n)
    model.eval(); correct = 0
    with torch.no_grad():
        for i in range(0, len(Xte), 500):
            correct += (model(Xte[i:i + 500]).argmax(1) == Yte[i:i + 500]).sum().item()
    return {'accuracy': correct / len(Xte), 'latency_ms': latency_ms(model), 'params': count_params(model), 'loss': curve}, model

COUNTER = [0]; BEST = {'acc': -1, 'model': None}

def evaluate_many(genos, gen, tag):
    out = []
    for g in genos:
        try:
            r = train_eval(g)
        except RuntimeError as e:                                         # e.g. CUDA out of memory
            print('  skipped:', str(e)[:60]); torch.cuda.empty_cache(); continue
        if r is None: continue
        res, model = r
        COUNTER[0] += 1
        out.append({'id': COUNTER[0], 'gen': gen, 'tag': tag, 'genotype': g.to_dict(), **res})
        if tag == 'evo' and res['accuracy'] > BEST['acc']:
            BEST['acc'], BEST['model'] = res['accuracy'], model.cpu()
        if dev.type == 'cuda': torch.cuda.empty_cache()
    return out

In [ ]:
evo, rnd, hist = [], [], []
best_evo = best_rnd = 0.0
pop = [random_genotype(SPACE) for _ in range(POP)]
t_start = time.time()

for gen in range(GENS):
    t0 = time.time()
    evo += evaluate_many(pop, gen, 'evo')
    rnd += evaluate_many([random_genotype(SPACE) for _ in range(POP)], gen, 'rnd')      # same budget, no evolution
    if evo:
        elite, ranks = nsga2_select(evo, POP)
        for t, r in zip(evo, ranks): t['pareto_rank'] = r
        pop = next_generation(elite, POP, SPACE, op_weights=adapt_weights(evo))
        best_evo = max(best_evo, max(t['accuracy'] for t in evo))
    else:
        pop = [random_genotype(SPACE) for _ in range(POP)]
    if rnd: best_rnd = max(best_rnd, max(t['accuracy'] for t in rnd))
    hist.append({'gen': gen + 1, 'evo': best_evo, 'rnd': best_rnd})
    print(f'gen {gen + 1}/{GENS}  evo best {best_evo:.3f}  random best {best_rnd:.3f}  ({time.time() - t0:.0f}s)')
print(f'total {(time.time() - t_start) / 60:.1f} min, {len(evo)} evolved + {len(rnd)} random candidates')

In [ ]:
combined = evo + rnd
_, cranks = nsga2_select(combined, len(combined))
front = [t for t, r in zip(combined, cranks) if r == 0]
share_evo = sum(t['tag'] == 'evo' for t in front)
print(f"Best accuracy   evolution {best_evo:.3f}   random {best_rnd:.3f}")
print(f"Combined Pareto front: {len(front)} designs, {share_evo} from evolution, {len(front) - share_evo} from random")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].plot([h['gen'] for h in hist], [h['evo'] for h in hist], 'o-', color='#3b82f6', label='evolution (NSGA-II)')
ax[0].plot([h['gen'] for h in hist], [h['rnd'] for h in hist], 's--', color='#9ca3af', label='random search')
ax[0].set_xlabel('generation (same budget)'); ax[0].set_ylabel('best accuracy so far'); ax[0].legend(); ax[0].grid(alpha=.3)
for pool, c, lab in ((rnd, '#9ca3af', 'random'), (evo, '#3b82f6', 'evolution')):
    ax[1].scatter([t['latency_ms'] for t in pool], [t['accuracy'] for t in pool], s=[20 + t['params'] / 20000 for t in pool], c=c, alpha=.6, label=lab)
ax[1].scatter([t['latency_ms'] for t in front], [t['accuracy'] for t in front], facecolors='none', edgecolors='k', s=140, label='combined Pareto front')
ax[1].set_xlabel('latency (ms, batch 1)'); ax[1].set_ylabel('accuracy'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig('benchmark.png', dpi=150); plt.show()

In [ ]:
# ---- export: session for the website, ONNX of the best model, summary ----
def site_block(b): return {'t': b['type'], 'c': b['channels'], 'k': b['kernel'], 'skip': bool(b['skip'])}
trials = [{'id': t['id'], 'gen': t['gen'], 'lr': t['genotype']['lr'], 'acc': t['accuracy'], 'lat': t['latency_ms'],
           'params': t['params'], 'loss': t['loss'], 'blocks': [site_block(b) for b in t['genotype']['blocks']]} for t in evo]
session = {'v': 5, 'source': 'colab', 'gen': GENS, 'target': 'Server GPU', 'popSize': POP, 'hist': hist, 'trials': trials,
           'meta': {'gpu': GPU, 'epochs': EPOCHS, 'train_n': TRAIN_N, 'test_n': TEST_N, 'seed': SEED, 'best_evo': best_evo, 'best_random': best_rnd}}
json.dump(session, open('session.json', 'w'))
if BEST['model'] is not None:
    torch.onnx.export(BEST['model'].eval(), torch.randn(1, 3, 32, 32), 'best.onnx', opset_version=17,
                      input_names=['input'], output_names=['logits'], dynamic_axes={'input': {0: 'batch'}})
summary = {'best_evolution': best_evo, 'best_random': best_rnd, 'combined_front': len(front), 'front_from_evolution': share_evo,
           'candidates_each': [len(evo), len(rnd)], 'settings': {'POP': POP, 'GENS': GENS, 'EPOCHS': EPOCHS, 'TRAIN_N': TRAIN_N}, 'gpu': GPU}
json.dump(summary, open('summary.json', 'w'), indent=2); print(json.dumps(summary, indent=2))
!zip -q deepoptima-results.zip session.json best.onnx summary.json benchmark.png
try:
    from google.colab import files; files.download('deepoptima-results.zip')
except Exception as e:
    print('download manually from the file browser:', e)

## Next
* Website -> **Dashboard -> Load session** -> pick `session.json` to explore the real run in the Pareto and loss charts.
* Put `benchmark.png` and the two numbers above into your README and LinkedIn post, with the settings (`POP`, `GENS`, `EPOCHS`, `TRAIN_N`) so the comparison is reproducible.
* For stronger results try `POP=16, GENS=10, EPOCHS=4, TRAIN_N=20000` (several hours on the free tier; Colab may disconnect).